In [2]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import webview
import os
import glob

In [10]:
def filter_minflux_data(file_path, verbose=True):
    """Load and filter MINFLUX data from a single file."""
    MFX_Data = np.load(file_path)
    if verbose:
        print(f'\n--- Processing: {os.path.basename(file_path)} ---')
        print('all raw                  ', len(MFX_Data))

    # Keep only valid localizations
    MFX_Data = MFX_Data[MFX_Data['vld'] == True]
    if verbose:
        print('only valid_              ', len(MFX_Data))

    # Keep only last iteration
    MFX_Data = MFX_Data[MFX_Data['itr'] == np.max(MFX_Data['itr'])]
    if verbose:
        print('last iteration only      ', len(MFX_Data))

    # Number of final localizations per trace
    unique_tids, inv_idx, locs_per_tid = np.unique(
        MFX_Data['tid'], return_inverse=True, return_counts=True
    )
    if verbose:
        print(f'min trace                 {np.min(locs_per_tid)}')

    # Keep only traces with at least 4 localizations
    MFX_Data = MFX_Data[locs_per_tid[inv_idx] > 3]
    if verbose:
        print('after trace length filt  ', len(MFX_Data))

    _, locs_per_tid_filt = np.unique(MFX_Data['tid'], return_counts=True)
    if verbose:
        print(f'Minimum trace             {np.min(locs_per_tid_filt)}')

    return MFX_Data

In [11]:
# -----------------------------
# Load + process both MINFLUX data files
# -----------------------------
path = '/Users/maximiliansenftleben/Data/8376_alm/8376_minflux/data/for Max_MFX_echange_2colour_2D/'
files = [i for i in glob.glob(os.path.join(path, "**", "*.npy"), recursive=True) if i.endswith('.npy')]

print(f"Found {len(files)} files:")
for f in files:
    print(f"  - {os.path.basename(f)}")

# Process both files
MFX_Data_1 = filter_minflux_data(files[0])

Found 2 files:
  - 241008-133841_NUP96_minflux.npy
  - 241008-144515_NUP153_minflux.npy

--- Processing: 241008-133841_NUP96_minflux.npy ---
all raw                   89537
only valid_               89537
last iteration only       21237
min trace                 1
after trace length filt   21048
Minimum trace             4


## DBSCAN: what it is and why we use it
DBSCAN (Density-Based Spatial Clustering of Applications with Noise) groups points by **local density** rather than by forcing every point into a cluster.

Key ideas:
- `eps`: neighborhood radius around each point.
- `min_samples`: minimum number of points inside `eps` to call a point a **core point**.
- Clusters grow by connecting density-reachable points from core points.
- Points that never reach cluster density are labeled as **noise** (`-1`).

Why this is useful here:
- You do not need to pre-define the number of clusters.
- It can discover irregular cluster shapes.
- It naturally detects outliers in localization data.

In the next cells, we will:
1. Extract XY coordinates from `MFX_Data_1`.
2. Standardize them so `eps` is easier to tune.
3. Run a NumPy implementation of DBSCAN.
4. Plot clusters and noise with Plotly.

In [12]:
# 1) Extract XY coordinates from the structured MINFLUX array
if not isinstance(MFX_Data_1, np.ndarray):
    raise TypeError("MFX_Data_1 must be a NumPy array.")

if MFX_Data_1.dtype.names is None:
    raise ValueError("MFX_Data_1 must be a structured array with named fields.")

field_names = list(MFX_Data_1.dtype.names)
print("Available fields:", field_names)

# Primary path for MINFLUX data: 'loc' contains (x, y, z)
if "loc" in field_names:
    loc = MFX_Data_1["loc"]
    if loc.ndim != 2 or loc.shape[1] < 2:
        raise ValueError(f"Unexpected 'loc' shape: {loc.shape}")
    x_field, y_field = "loc[:,0]", "loc[:,1]"
    XY = np.column_stack([loc[:, 0], loc[:, 1]]).astype(float)
else:
    # Fallback for other table formats with explicit x/y fields
    x_candidates = ["x", "x_nm", "locx", "xn", "px", "x_pos"]
    y_candidates = ["y", "y_nm", "locy", "yn", "py", "y_pos"]

    x_field = next((f for f in x_candidates if f in field_names), None)
    y_field = next((f for f in y_candidates if f in field_names), None)

    if x_field is None or y_field is None:
        raise KeyError(
            "Could not find X/Y fields automatically. "
            f"Please set x_field and y_field manually from: {field_names}"
        )

    XY = np.column_stack([MFX_Data_1[x_field].astype(float), MFX_Data_1[y_field].astype(float)])

print(f"Using fields: x='{x_field}', y='{y_field}'")
print(f"Number of points: {XY.shape[0]}")
print("First 5 points:\n", XY[:5])

Available fields: ['vld', 'fnl', 'bot', 'eot', 'sta', 'tim', 'tid', 'gri', 'thi', 'sqi', 'itr', 'loc', 'lnc', 'eco', 'ecc', 'efo', 'efc', 'fbg', 'cfr', 'dcr']
Using fields: x='loc[:,0]', y='loc[:,1]'
Number of points: 21048
First 5 points:
 [[6.68113933e-06 1.59464785e-07]
 [6.67789170e-06 1.64654116e-07]
 [6.67361723e-06 1.68059178e-07]
 [6.66538798e-06 1.64982283e-07]
 [6.67240479e-06 1.62260138e-07]]


In [13]:
# 2) Standardize coordinates and run DBSCAN (pure NumPy implementation)
# Standardization makes eps interpretable in "number of standard deviations".
mu = XY.mean(axis=0)
sigma = XY.std(axis=0)
sigma[sigma == 0] = 1.0
XY_scaled = (XY - mu) / sigma

print("Coordinate mean:", mu)
print("Coordinate std:", sigma)


def dbscan_numpy(points, eps=0.25, min_samples=10):
    """Simple DBSCAN implementation using a precomputed distance matrix."""
    n_points = points.shape[0]
    labels = np.full(n_points, -99, dtype=int)  # -99 means unvisited

    # Pairwise Euclidean distances
    diff = points[:, None, :] - points[None, :, :]
    dist = np.sqrt(np.sum(diff**2, axis=2))

    # Neighborhood indices for each point
    neighbors = [np.where(dist[i] <= eps)[0] for i in range(n_points)]

    cluster_id = 0

    for i in range(n_points):
        if labels[i] != -99:
            continue

        if len(neighbors[i]) < min_samples:
            labels[i] = -1  # noise
            continue

        labels[i] = cluster_id
        seeds = [j for j in neighbors[i] if j != i]
        seed_set = set(seeds)

        while seeds:
            j = seeds.pop(0)

            if labels[j] == -1:
                labels[j] = cluster_id

            if labels[j] != -99:
                continue

            labels[j] = cluster_id

            if len(neighbors[j]) >= min_samples:
                for k in neighbors[j]:
                    if k not in seed_set:
                        seeds.append(k)
                        seed_set.add(k)

        cluster_id += 1

    labels[labels == -99] = -1
    return labels

# Tune these two parameters for your dataset
EPS = 0.18
MIN_SAMPLES = 12

labels = dbscan_numpy(XY_scaled, eps=EPS, min_samples=MIN_SAMPLES)

n_noise = int(np.sum(labels == -1))
cluster_ids = sorted([cid for cid in np.unique(labels) if cid != -1])

print(f"DBSCAN finished with eps={EPS}, min_samples={MIN_SAMPLES}")
print(f"Clusters found: {len(cluster_ids)}")
print(f"Noise points: {n_noise} / {len(labels)}")
for cid in cluster_ids:
    print(f"  Cluster {cid}: {(labels == cid).sum()} points")

Coordinate mean: [7.38148446e-06 9.83046593e-07]
Coordinate std: [6.00495097e-07 5.77633532e-07]
DBSCAN finished with eps=0.18, min_samples=12
Clusters found: 40
Noise points: 35 / 21048
  Cluster 0: 746 points
  Cluster 1: 4723 points
  Cluster 2: 706 points
  Cluster 3: 987 points
  Cluster 4: 525 points
  Cluster 5: 406 points
  Cluster 6: 885 points
  Cluster 7: 544 points
  Cluster 8: 35 points
  Cluster 9: 2788 points
  Cluster 10: 859 points
  Cluster 11: 1639 points
  Cluster 12: 13 points
  Cluster 13: 14 points
  Cluster 14: 448 points
  Cluster 15: 770 points
  Cluster 16: 105 points
  Cluster 17: 1027 points
  Cluster 18: 21 points
  Cluster 19: 249 points
  Cluster 20: 470 points
  Cluster 21: 560 points
  Cluster 22: 541 points
  Cluster 23: 38 points
  Cluster 24: 252 points
  Cluster 25: 27 points
  Cluster 26: 310 points
  Cluster 27: 27 points
  Cluster 28: 468 points
  Cluster 29: 13 points
  Cluster 30: 123 points
  Cluster 31: 36 points
  Cluster 32: 166 points
  C

In [14]:
# 3) Plot DBSCAN result with Plotly
unique_labels = sorted(np.unique(labels))

# Distinct color palette for clusters; noise is forced to gray
palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
    "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#4c78a8", "#f58518"
]

fig = go.Figure()

for i, lab in enumerate(unique_labels):
    mask = labels == lab
    pts = XY[mask]

    if lab == -1:
        name = f"Noise ({pts.shape[0]})"
        color = "#9e9e9e"
        marker_size = 4
        opacity = 0.45
    else:
        name = f"Cluster {lab} ({pts.shape[0]})"
        color = palette[i % len(palette)]
        marker_size = 5
        opacity = 0.8

    fig.add_trace(
        go.Scattergl(
            x=pts[:, 0],
            y=pts[:, 1],
            mode="markers",
            name=name,
            marker=dict(size=marker_size, color=color, opacity=opacity),
        )
    )

fig.update_layout(
    title=f"DBSCAN clustering of MINFLUX points (eps={EPS}, min_samples={MIN_SAMPLES})",
    xaxis_title=f"{x_field}",
    yaxis_title=f"{y_field}",
    template="plotly_white",
    legend_title="Label",
    width=950,
    height=700,
)

fig.show()